# 06 — Connect CRANE modules to direct eRegulons

**Objective.** Convert group support aggregates into rank matrices, module-to-eRegulon edges, and compact PBMC biological summaries.

CRANE-GRN does not replace the SCENIC+ regulatory backbone. It adds an upstream factor-combination layer: **factor combination → regulatory region → existing eRegulon**.

## Configuration and imports

In [ ]:
import gc
import os
from pathlib import Path

import matplotlib.pyplot as plt
import mudata
import numpy as np
import pandas as pd

from _tutorial_utils import (
    PBMC_GROUP_ORDER,
    _EXPECTED_GROUPS,
    classify_module_lineage,
    ensure_crane_grn_importable,
    load_public_aggregate,
    plot_heatmap,
    plot_module_overlay_network,
    prepare_module_overlay_network,
    require_path,
    safe_group_name,
)

ensure_crane_grn_importable()

from crane_grn.eregulon import (
    build_module_eregulon_edges,
    build_support_rank_matrices,
    summarize_modules,
)
from crane_grn.io import get_modality

DATA_ROOT = Path(os.environ.get("CRANE_GRN_DATA_ROOT","../../CRANE_GRN_data",)).expanduser().resolve()
WORK_ROOT = Path(os.environ.get("CRANE_GRN_TUTORIAL_WORK","./results",)).expanduser().resolve()
PBMC_INPUT_ROOT = DATA_ROOT / "PBMC" / "prepared_inputs"
PBMC_REFERENCE_ROOT = DATA_ROOT / "PBMC" / "outputs"
SUPPORT_ROOT = WORK_ROOT / "05_support"
RESPONSIBILITY_ROOT = WORK_ROOT / "03_responsibility"
OUTPUT_ROOT = WORK_ROOT / "06_eregulons"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Resolve generated or frozen aggregates

Generated aggregates are preferred. Frozen manuscript aggregates are used as a read-only fallback so the connection layer can be inspected independently. Each file is reopened and validated against the exact six-field public schema.

In [ ]:
generated_aggregates = sorted(SUPPORT_ROOT.glob("agg_*.npz")) if SUPPORT_ROOT.exists() else []
if generated_aggregates:
    aggregate_paths = generated_aggregates
    aggregate_source = "generated tutorial outputs"
else:
    aggregate_paths = sorted(PBMC_REFERENCE_ROOT.glob("agg_*.npz"))
    aggregate_source = "frozen manuscript references"
if len(aggregate_paths) != 13:
    raise ValueError("Expected 13 PBMC aggregate files; found {}.".format(len(aggregate_paths)))
print("Using", aggregate_source)

scplus = mudata.read(str(require_path(PBMC_INPUT_ROOT / "scplusmdata.h5mu", "PBMC scplusmdata input")))
atac = get_modality(scplus, "scATAC_counts")
actual_groups = sorted(atac.obs["cell_type"].astype(str).unique().tolist())
group_lookup = {safe_group_name(group): group for group in actual_groups}

def group_from_aggregate_path(path):
    token = path.stem[len("agg_"):]
    if token in actual_groups:
        return token
    if token in group_lookup:
        return group_lookup[token]
    raise ValueError("Could not recover a PBMC group label from {}.".format(path.name))

## Rank matrices, edges, and summaries

The triplet filter follows the executed manuscript logic. Within each eRegulon, modules are ranked by support; within each module, eRegulons and their positive regulatory triplets are ranked with the SCENIC+ triplet rank used for tie resolution. Edges retain up to five positive-support modules per eRegulon.

In [ ]:
responsibility_triplet = pd.read_parquet(require_path(RESPONSIBILITY_ROOT / "responsibility_triplet.parquet", "tutorial triplet responsibility"))
eregulon_metadata = pd.read_csv(require_path(PBMC_REFERENCE_ROOT / "eRegulon_direct.tsv", "direct eRegulon metadata"), sep="\t")
support_profiles = []
edge_tables = []
module_summaries = []

for aggregate_path in aggregate_paths:
    group = group_from_aggregate_path(aggregate_path)
    aggregate = load_public_aggregate(aggregate_path)
    modules = aggregate["modules"].astype(str)
    eregulons = aggregate["eregs"].astype(str)
    support = pd.DataFrame(aggregate["support_mean_KE"], index=modules, columns=eregulons)
    correlation = pd.DataFrame(aggregate["corr_auc_sup_KE"], index=modules, columns=eregulons)
    if support.shape != (530, 113):
        raise ValueError("{} has support shape {}; expected (530, 113).".format(aggregate_path.name, support.shape))

    ranks = build_support_rank_matrices(
        support_by_eregulon=support,
        eregulon_metadata=eregulon_metadata,
        region_universe=responsibility_triplet.index,
    )
    edges = build_module_eregulon_edges(
        support_mean=support,
        corr_auc_sup=correlation,
        modules=modules,
        eregulons=eregulons,
        responsibility_triplet=responsibility_triplet,
        eregulon_metadata=eregulon_metadata,
        top_k=5,
        support_minimum=0.0,
    )
    edges.insert(0, "group", group)
    summary = summarize_modules(edges.drop(columns="group"), group=group)

    group_root = OUTPUT_ROOT / safe_group_name(group)
    group_root.mkdir(parents=True, exist_ok=True)
    ranks["module_ranking_per_eregulon"].to_parquet(group_root / "module_ranking_per_eregulon.parquet")
    ranks["eregulon_ranking_per_module"].to_parquet(group_root / "eregulon_ranking_per_module.parquet")
    ranks["module_ranking_per_triplet"].to_parquet(group_root / "module_ranking_per_triplet.parquet")
    ranks["triplet_ranking_per_module"].to_parquet(group_root / "triplet_ranking_per_module.parquet")
    edges.to_csv(group_root / "module_eregulon_edges.tsv", sep="\t", index=False)
    summary.to_csv(group_root / "module_summary.tsv", sep="\t", index=False)

    support_profiles.append(support.mean(axis=1).rename(group))
    edge_tables.append(edges)
    module_summaries.append(summary)
    del aggregate, support, correlation, ranks, edges, summary
    gc.collect()

support_by_group = pd.concat(support_profiles, axis=1).T
all_edges = pd.concat(edge_tables, ignore_index=True)
all_module_summaries = pd.concat(module_summaries, ignore_index=True)
support_by_group.to_parquet(OUTPUT_ROOT / "group_by_module_mean_support.parquet")
all_edges.to_csv(OUTPUT_ROOT / "all_module_eregulon_edges.tsv", sep="\t", index=False)
all_module_summaries.to_csv(OUTPUT_ROOT / "all_module_summaries.tsv", sep="\t", index=False)
print("Combined edge table:", all_edges.shape)

## Compact PBMC examples

The heatmap selects up to three high-support modules from each unambiguous PBMC lineage class. It is a descriptive tutorial view, not a significance analysis.

In [ ]:
module_lineage = pd.Series(
    {
        module: classify_module_lineage(module)
        for module in support_by_group.columns
    },
    name="lineage",
)

selected_modules = []
selection_records = []

for lineage in ["T/NK", "B", "Myeloid"]:
    lineage_modules = [
        module
        for module in module_lineage.index[
            module_lineage == lineage
        ]
        if module in support_by_group.columns
    ]

    expected_groups = [
        group
        for group in _EXPECTED_GROUPS[lineage]
        if group in support_by_group.index
    ]
    other_groups = [
        group
        for group in support_by_group.index
        if group not in expected_groups
    ]

    if (
        not lineage_modules
        or not expected_groups
        or not other_groups
    ):
        continue

    expected_mean = support_by_group.loc[
        expected_groups,
        lineage_modules,
    ].mean(axis=0)

    other_mean = support_by_group.loc[
        other_groups,
        lineage_modules,
    ].mean(axis=0)

    specificity = (
        expected_mean - other_mean
    ).sort_values(ascending=False)

    selected = specificity.loc[
        specificity > 0
    ].head(3).index.tolist()

    selected_modules.extend(selected)

    for module in selected:
        selection_records.append(
            {
                "lineage": lineage,
                "module": module,
                "expected_support_mean": float(
                    expected_mean[module]
                ),
                "other_support_mean": float(
                    other_mean[module]
                ),
                "specificity_delta": float(
                    specificity[module]
                ),
            }
        )

selected_modules = list(dict.fromkeys(selected_modules))

if not selected_modules:
    raise ValueError(
        "No lineage-specific modules were available for the heatmap."
    )

module_selection = pd.DataFrame(selection_records)
display(module_selection)

present_order = [
    group
    for group in PBMC_GROUP_ORDER
    if group in support_by_group.index
]

heatmap_raw = support_by_group.loc[
    present_order,
    selected_modules,
].copy()

module_mean = heatmap_raw.mean(axis=0)
module_std = heatmap_raw.std(
    axis=0,
    ddof=0,
).replace(0, np.nan)

heatmap_z = (
    heatmap_raw - module_mean
).divide(module_std, axis=1)

heatmap_z = heatmap_z.dropna(axis=1, how="all")

color_limit = float(
    np.nanmax(
        np.abs(
            heatmap_z.to_numpy(dtype=float)
        )
    )
)

plot_heatmap(
    heatmap_z,
    title=(
        "Cell-type-relative support of "
        "lineage-associated PBMC modules"
    ),
    colorbar_label="Support z-score across cell types",
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
    x_rotation=60,
)

plt.tight_layout()
plt.show()

## Illustrative module–eRegulon network

The two panels use identical nodes and radial coordinates.

- **SCENIC+ eRegulon union:** eRegulon–region–target-gene structure associated
  with the selected module.
- **CRANE module overlay:** the same structure with the factor-combination
  module added at the center and positive-responsibility regions highlighted.

The network does not compare biological conditions. It shows what CRANE-GRN adds
to the corresponding SCENIC+ eRegulon structure.

In [ ]:
ILLUSTRATIVE_GROUP = "NK"
ILLUSTRATIVE_MODULE = "ARID1B;BCL11B;GATA3"

illustrative_summary = all_module_summaries.loc[
    (
        all_module_summaries["group"].astype(str)
        == ILLUSTRATIVE_GROUP
    )
    & (
        all_module_summaries["module"].astype(str)
        == ILLUSTRATIVE_MODULE
    )
].copy()

if illustrative_summary.empty:
    raise ValueError(
        "The canonical PBMC illustrative group/module combination "
        "was not found in all_module_summaries."
    )

if int(
    illustrative_summary.iloc[0]["n_resp_total"]
) <= 0:
    raise ValueError(
        "The illustrative module has no responsible regions."
    )

illustrative_edges = all_edges.loc[
    all_edges["group"].astype(str)
    == ILLUSTRATIVE_GROUP
].copy()

network = prepare_module_overlay_network(
    edge_table=illustrative_edges,
    module=ILLUSTRATIVE_MODULE,
    responsibility_triplet=responsibility_triplet,
    eregulon_metadata=eregulon_metadata,
    max_responsible_regions=10,
    max_background_regions=15,
    random_seed=42,
)

display(
    illustrative_summary[
        [
            "group",
            "module",
            "n_ereg",
            "support_mean",
            "support_max",
            "corr_mean",
            "corr_positive_fraction",
            "n_resp_total",
        ]
    ]
)

responsible_regions_shown = set().union(
    *[
        branch["responsible_regions"]
        for branch in network["branches"].values()
    ]
)

print("Illustrative group:", ILLUSTRATIVE_GROUP)
print("Illustrative module:", ILLUSTRATIVE_MODULE)
print("Displayed eRegulons:", len(network["branches"]))
print(
    "Displayed responsible regions:",
    len(responsible_regions_shown),
)

figure, axes = plot_module_overlay_network(
    network,
    output_path=(
        OUTPUT_ROOT
        / "illustrative_module_eregulon_network.png"
    ),
)

plt.show()

## Interpretation and outputs

The final edge table links a selected factor-combination module to an existing direct eRegulon when the module has positive support over regulatory regions assigned to that eRegulon. Correlation with region-based AUCell is retained as an independent descriptive field.

A one-group dataset produces one aggregate file; all rank and edge APIs remain unchanged. Outputs are stored by group under `WORK_ROOT/06_eregulons`, with combined edge and summary tables at the directory root.

In [ ]:
del responsibility_triplet, eregulon_metadata, support_profiles
del edge_tables, module_summaries, atac, scplus
gc.collect()